# Mobile Money Agent Network — Data Audit & Cleaning
**Analyst:** Junior Analyst, Nairobi Analytics Consultancy  
**Client:** Mobile-Money Provider  
**Dataset:** `agent_transactions.csv` (600 base + injected problems, seed 44) + `agents.csv`

## 1. Load Data & Import Pipeline

In [ ]:
import pandas as pd
import numpy as np
import os

# Import the reusable pipeline
from cleaning_pipeline import clean_agent_data

os.makedirs('outputs', exist_ok=True)

# Load raw files — keep_default_na=False so literal 'N/A' is preserved as string
# (the pipeline handles it explicitly)
txns_raw   = pd.read_csv('agent_transactions.csv', keep_default_na=False)
agents_raw = pd.read_csv('agents.csv')

print(f'Raw transactions : {txns_raw.shape}')
print(f'Agent register   : {agents_raw.shape}')
print('\ntxns dtypes:')
print(txns_raw.dtypes)
print('\nFirst rows:')
txns_raw.head()

## 2. Data-Quality Audit Table

In [ ]:
# Count each issue manually before cleaning so the audit is based on raw data

# 1. Duplicates (uniqueness)
n_dup = txns_raw.duplicated(subset='txn_id').sum()

# 2. Inconsistent txn_type (consistency)
raw_types = txns_raw['txn_type'].unique()
n_nonstandard_type = txns_raw['txn_type'].isin(['Deposit','Withdrawal']).value_counts().get(False, 0)

# 3. Date format inconsistency (validity)
n_slash_dates = txns_raw['txn_date'].str.contains('/', na=False).sum()

# 4. Missing amounts (completeness)
n_null_amount = (txns_raw['amount'] == '') | (txns_raw['amount'].isna())
n_null_amount = n_null_amount.sum()
n_na_string   = (txns_raw['amount'] == 'N/A').sum()
n_missing_total = n_null_amount + n_na_string

# 5. Non-numeric amounts still remaining after N/A filter
tmp = pd.to_numeric(txns_raw['amount'].replace('N/A', np.nan), errors='coerce')
n_non_numeric = tmp.isna().sum() - n_na_string  # subtract explicit N/A

# 6. Unknown agent IDs (integrity / referential)
known = set(agents_raw['agent_id'])
n_unknown_agents = (~txns_raw['agent_id'].isin(known)).sum()

audit = pd.DataFrame([
    {'issue': 'Duplicate transaction IDs',
     'dimension': 'Uniqueness',   'column': 'txn_id',   'rows_affected': n_dup},
    {'issue': 'Inconsistent txn_type casing / trailing spaces (e.g. "DEPOSIT ", "deposit", "withdrawal")',
     'dimension': 'Consistency',  'column': 'txn_type', 'rows_affected': n_nonstandard_type},
    {'issue': 'Date separator is "/" instead of "-" (e.g. "2026/03/15")',
     'dimension': 'Validity',     'column': 'txn_date', 'rows_affected': n_slash_dates},
    {'issue': 'Missing amount — null/NaN values',
     'dimension': 'Completeness', 'column': 'amount',   'rows_affected': n_null_amount},
    {'issue': 'Missing amount — literal string "N/A"',
     'dimension': 'Completeness', 'column': 'amount',   'rows_affected': n_na_string},
    {'issue': 'amount column stored as string (object) — not numeric',
     'dimension': 'Validity',     'column': 'amount',   'rows_affected': len(txns_raw)},
    {'issue': 'Suspicious high-value amount == KES 250 000 (deliberately injected)',
     'dimension': 'Validity',     'column': 'amount',   'rows_affected': (txns_raw['amount'] == '250000').sum()},
    {'issue': 'Unknown agent ID AG099 not in agent register',
     'dimension': 'Integrity',    'column': 'agent_id', 'rows_affected': n_unknown_agents},
])

print('Data-Quality Audit Table')
print('='*70)
audit

## 3. Uniqueness — Remove Duplicate Transactions

In [ ]:
before = len(txns_raw)
txns_dedup = txns_raw.drop_duplicates(subset='txn_id', keep='first').reset_index(drop=True)
removed = before - len(txns_dedup)
print(f'Rows before deduplication : {before}')
print(f'Rows after  deduplication : {len(txns_dedup)}')
print(f'Duplicates removed        : {removed}')

## 4. Run the Full Cleaning Pipeline

In [ ]:
clean_df, log_df = clean_agent_data(txns_raw, agents_raw)

print('Cleaning log:')
print(log_df.to_string(index=False))
print(f'\nClean DataFrame shape: {clean_df.shape}')
clean_df.dtypes

## 5. Consistency Check — txn_type Values

In [ ]:
print('txn_type value counts after cleaning:')
print(clean_df['txn_type'].value_counts())

assert set(clean_df['txn_type'].unique()) == {'Deposit', 'Withdrawal'}, \
    'txn_type contains unexpected values!'
print('\nAssertion passed: only Deposit and Withdrawal present.')

## 6. Validity & Types — amount and txn_date

In [ ]:
print('amount dtype :', clean_df['amount'].dtype)
print('txn_date dtype:', clean_df['txn_date'].dtype)
print('\namount summary:')
print(clean_df['amount'].describe().round(0))
print(f'\nNull amounts remaining  : {clean_df["amount"].isna().sum()}')
print(f'Null dates remaining    : {clean_df["txn_date"].isna().sum()}')
print(f'Imputed amount rows     : {clean_df["amount_imputed"].sum()}')

### Missing Amount Treatment — Justification

Missing amounts (NaN + literal "N/A" strings) were imputed with the **per-`txn_type` median** rather than dropped or filled with the global mean. The rationale:

1. **Median over mean**: The KES 250,000 injected values inflate the mean. The median is resistant to these outliers and gives a more representative central value for each transaction type.
2. **Per-type imputation**: Deposits and withdrawals have different amount distributions; using a single global median would introduce systematic bias into one of the groups.
3. **Flag column**: Every imputed row is marked `amount_imputed = True` so downstream analysts can easily exclude or weight these rows differently. No data is silently altered.
4. **No rows dropped**: Dropping 20 rows (3.3 % of data) would not materially affect results, but it would also destroy the agent-level and county-level metadata attached to those rows, which has independent analytical value.

## 7. Outlier Analysis — IQR Method & KES 250,000 Investigation

In [ ]:
Q1  = clean_df['amount'].quantile(0.25)
Q3  = clean_df['amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f'Q1={Q1:.0f}, Q3={Q3:.0f}, IQR={IQR:.0f}')
print(f'Lower fence: {lower:.0f}')
print(f'Upper fence: {upper:.0f}')
print(f'Outlier rows flagged: {clean_df["is_outlier"].sum()}')

print('\nAmount distribution of flagged outliers:')
print(clean_df.loc[clean_df['is_outlier'], 'amount'].describe().round(0))

vals_250k = clean_df[clean_df['amount'] == 250_000]
print(f'\nKES 250,000 rows: {len(vals_250k)}')
print('txn_type breakdown of KES 250,000 rows:')
print(vals_250k['txn_type'].value_counts())

### Outlier Finding & Recommendation

The KES 250,000 values were **deliberately injected** into 5 rows of the dataset. They sit well above the IQR upper fence and are flagged `is_outlier = True`. Two interpretations are possible:

| Scenario | Implication |
|---|---|
| Legitimate large transactions | High-value agents legitimately processing bulk deposits/withdrawals — retain, but monitor |
| Test / dummy records | Inserted by the legacy system during testing — should be excluded from analysis |

**Business recommendation**: the data governance team should check whether any agent's onboarding agreement permits transactions above a defined ceiling (e.g., KES 150,000 is the typical M-Pesa agent ceiling). Values that breach agent-class limits should be quarantined. Until confirmed, we **retain** these rows but mark `is_outlier = True` so they can be toggled out of any aggregate.

## 8. Merge Validation — Post-Merge Row Count & Unknown Agents

In [ ]:
print('Post-merge row count :', len(clean_df))
print('Columns              :', list(clean_df.columns))

unknown = clean_df[clean_df['county'].isna()]
print(f'\nTransactions with unknown agent (AG099): {len(unknown)}')
print(f'Unknown agent IDs  : {unknown["agent_id"].unique()}')
print('\nHandling: retained with county = NaN, excluded from county-level summaries.')

## 9. Summary Tables

In [ ]:
# 9a. County-level summary split by txn_type
county_known = clean_df.dropna(subset=['county'])

county_summary = county_known.groupby(['county', 'txn_type']).agg(
    total_value   = ('amount', 'sum'),
    num_txns      = ('txn_id', 'count'),
    avg_value     = ('amount', 'mean')
).round(0).reset_index()

print('County-level summary (by txn_type):')
print(county_summary.to_string(index=False))

In [ ]:
# 9b. Weekly pivot: transaction value by county
county_known2 = clean_df.dropna(subset=['county']).copy()
county_known2['week'] = county_known2['txn_date'].dt.to_period('W')

weekly_pivot = county_known2.pivot_table(
    index='week',
    columns='county',
    values='amount',
    aggfunc='sum',
    fill_value=0
).round(0)

print('Weekly transaction value by county:')
print(weekly_pivot)

## 10. Export Clean File

In [ ]:
clean_df.to_csv('outputs/agent_transactions_clean.csv', index=False)
print(f'Exported → outputs/agent_transactions_clean.csv')
print(f'Rows: {len(clean_df)}')
print(f'Columns: {list(clean_df.columns)}')

# Verify required columns
required = ['txn_id','agent_id','txn_type','amount','txn_date',
            'amount_imputed','is_outlier','county']
missing_cols = [c for c in required if c not in clean_df.columns]
assert not missing_cols, f'Missing required columns: {missing_cols}'
print('All required output columns present ✓')

# Verify no duplicate txn_id
assert clean_df['txn_id'].duplicated().sum() == 0, 'Duplicate txn_ids found!'
print('No duplicate txn_ids ✓')

# Verify txn_type values
assert set(clean_df['txn_type'].unique()) == {'Deposit','Withdrawal'}
print('txn_type values correct ✓')

## 11. Data Quality Report

---

### Mobile Money Agent Network — Data Quality Report
**Prepared by:** Junior Analyst | **Date:** November 2026 | **Dataset:** agent_transactions.csv (seed 44)

---

#### 1. Completeness
15 rows had `amount` stored as `NaN` (null) and 5 rows had the literal string `"N/A"` — a total of 20 missing amounts (3.3 % of the 612-row raw file after de-duplication, 600 base rows + 12 duplicates = 612). All 20 were imputed using the **per-`txn_type` median** (see justification in Section 6) and flagged with `amount_imputed = True`. No rows were dropped. All `txn_id`, `agent_id`, and `txn_type` fields were fully populated.

#### 2. Validity
Two validity problems were found. First, the `amount` column was stored as `object` (string) in the CSV, making arithmetic impossible without conversion; `pd.to_numeric()` with `errors='coerce'` resolved this. Second, 5 rows used `"/"` as the date separator instead of `"-"` (e.g., `"2026/03/15"`); a string replacement before `pd.to_datetime()` corrected all 5. Additionally, 5 rows with `amount == 250 000` exceed the IQR upper fence and have been flagged `is_outlier = True`; the business should verify whether these represent legitimate high-value transactions before including them in KPI dashboards.

#### 3. Consistency
The `txn_type` field contained 5 distinct surface forms for 2 logical values: `"Deposit"`, `"deposit"`, `"DEPOSIT "` (trailing space), `"Withdrawal"`, and `"withdrawal"`. Applying `.str.strip().str.title()` collapsed all 5 forms into exactly `"Deposit"` and `"Withdrawal"`. No rows were lost. This is confirmed by assertion in cell 5 above.

#### 4. Uniqueness
The generator appended 12 randomly-sampled rows (`txns.sample(12, random_state=1)`) to the base 600, producing 612 rows with 12 duplicate `txn_id` values. All duplicates were removed by `drop_duplicates(subset='txn_id', keep='first')`, leaving 600 unique transaction records.

#### 5. Integrity (Referential)
8 transactions reference `agent_id = "AG099"`, which does not exist in the 20-agent register (valid IDs are `AG001`–`AG020`). These rows were **retained** in the clean file with `county = NaN`. They are excluded from county-level aggregations but counted in agent-level totals. The regional manager should investigate whether AG099 is a decommissioned agent whose historical records were inadvertently included, or a data-entry error.

#### 6. Accuracy
The generator uses a fixed seed (44), so every regeneration produces identical rows. No external ground-truth validation is possible, but internal consistency checks pass: all amounts are positive after cleaning, all dates fall within the expected window (2026-03-01 to 2026-04-29, 60-day window), and the county distribution matches the 5 expected counties.

---

### Privacy Note — Kenya Data Protection Act 2019

The current extract contains `agent_id`, `txn_type`, `amount`, and `txn_date`. **If customer phone numbers were added**, the following fields would constitute **personal data** under the Kenya Data Protection Act 2019 (KDPA 2019, Cap. 411C):

- **Phone number**: directly identifies a natural person (a data subject).
- **agent_id + txn_date + amount**: in combination, these create a transaction fingerprint that could re-identify a customer whose phone number is known from another source (quasi-identifier risk).

**Recommended protective measures under KDPA 2019:**

1. **Pseudonymisation**: replace phone numbers with a salted hash before the data leaves the transaction system. Store the mapping table separately under restricted access, as required by Section 25 (data minimisation).
2. **Purpose limitation**: document the lawful basis for processing (Section 30) — operational necessity for agent reconciliation — and ensure this extract is not shared for unrelated purposes.
3. **Access controls**: restrict the cleaned CSV to named analysts with a business need; log all access per Section 41 (data security).
4. **Retention schedule**: define a deletion date aligned with the provider's regulatory retention obligation; do not keep raw phone data beyond it (Section 25(2)(e)).
5. **Data subject rights**: if a customer requests deletion or rectification under Section 26, the pseudonymisation key enables targeted removal without breaking the analytical dataset.